# Sample notebook for testing the ingestion module

This notebook demonstrates how to import and run the document ingestion workflow from the RAG package.

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:


import importlib
import logging
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'services').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return start


def ensure_dependencies() -> None:
    req_file = repo_root / 'requirements.txt'
    if req_file.exists():
        print('Installing project dependencies from requirements.txt...')
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade',
            '--prefer-binary',
            '-r',
            str(req_file),
        ])
        print('Dependencies installed.')
    else:
        print('requirements.txt not found; skipping dependency install.')


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

ensure_dependencies()

venv_python = repo_root / '.venv' / 'Scripts' / 'python.exe'
if venv_python.exists():
    os.environ['VIRTUAL_ENV'] = str(repo_root / '.venv')
    os.environ['PATH'] = f"{repo_root / '.venv' / 'Scripts'};{os.environ.get('PATH', '')}"

# Make notebook output show logging.info messages as well as print() output.
logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.basicConfig(level=logging.WARNING, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.basicConfig(level=logging.ERROR, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s:%(name)s:%(message)s', stream=sys.stdout)
logging.getLogger().setLevel(logging.INFO)
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
#logging.getLogger().setLevel(logging.DEBUG)

import services.rag.models as models_module
import services.rag.embeddings as embeddings_module
import services.rag.injest as injest_module
#import services.rag.visualization as visualization_module

for module in [models_module, embeddings_module, injest_module]:#, visualization_module]:
    try:
        importlib.reload(module)
    except Exception as exc:
        print(f'Reload warning for {module.__name__}: {exc}')

Injestor = injest_module.Injestor
#ChunkEmbeddingAnalyzer = visualization_module.ChunkEmbeddingAnalyzer

print('Imported Injestor successfully')
print(f'Notebook working directory: {repo_root}')

Installing project dependencies from requirements.txt...
Dependencies installed.
Imported Injestor successfully
Notebook working directory: C:\Rohit\Trainings\repo\FinancialAnalystCopilot


In [ ]:
import asyncio
from pathlib import Path

sample_dir = repo_root / 'notebooks' / 'sample_docs'
sample_dir.mkdir(parents=True, exist_ok=True)

sample_file = sample_dir / 'sample.txt'
sample_file.write_text(
    'The quick brown fox jumps over the lazy dog. ' * 20,
    encoding='utf-8',
)

print(f'Sample file created at: {sample_file}')

In [ ]:
async def run_ingestion():
    financeDocs = repo_root / 'notebooks' / 'sample_docs'
    #print(os.getenv("OPENAI_API_KEY"))
    use_external = os.getenv("OPENAI_API_KEY") is not None and os.getenv("OPENAI_API_KEY", "").strip() != ""
    if use_external:
        ingestor = Injestor(
            source_dir=financeDocs,
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            provider="openai",
            use_external_model=True,
            use_openai_embeddings=True
        )
        print("Using external OpenAI-compatible model via LiteLLM")
    else:
        ingestor = Injestor(source_dir=financeDocs, model='gemma3:270m')
        print("Using local Ollama model")

    print(f'Ingesting documents from: {financeDocs}')
    results = await ingestor.ingest_documents()
    return results

results = await run_ingestion()
print(f'Processed {len(results)} document(s)')
for result in results:
    print(f"Document: {result.document_name}")
    print(f"Chunks: {result.chunk_count}")
    for chunk in result.chunks[:3]:
        print(' -', chunk.headline, '|', chunk.summary, '| words:', chunk.word_count, '| text:', chunk.chunk_text[:50], '...')
    print()

In [ ]:
from services.rag.embeddings import OpenAIEmbeddingProvider
embedding_provider = OpenAIEmbeddingProvider()
textToEmbed = "Sample text to generate embeddings for"
print(embedding_provider.embed_texts([textToEmbed]))


# from openai import OpenAI
# openai = OpenAI()
# emb = openai.embeddings.create(model="text-embedding-3-large", input=textToEmbed).data
# print(emb)

In [ ]:
#create a sample list of ChunkRecord objects, with a collection of metadata_facts for each ChunkRecord
chunk_records = []
for i in range(5):
    chunk_record = models_module.ChunkRecord(
        chunk_text=f"Sample chunk text {i}",
        word_count=10 + i,
        headline=f"Sample headline {i}",
        document_name=f"Sample document {i}",
        chunk_index=i,
        summary=f"Sample summary {i}",
        metadata_facts=[]
    )
    chunk_records.append(chunk_record)

#write code to fetch the metadata_facts as dictionary from the chunk_records and print them
for chunk_record in chunk_records:
    metadata_facts_dict = {fact.key: fact.value for fact in chunk_record.metadata_facts}
    print(metadata_facts_dict)

metadatas = [
    {
        "document_name": chunk.document_name,
        "chunk_index": chunk.chunk_index,
        "word_count": chunk.word_count,
        "headline": chunk.headline,
        "summary": chunk.summary,
        "metadata_facts": {fact.key: fact.value for fact in chunk.metadata_facts},
    }
    for chunk in chunk_records
]

for metadata in metadatas:
    print(metadata)

In [2]:
#inspect chroma vectors

from cmath import sqrt
from pathlib import Path

import chromadb
from chromadb.config import Settings

DB_PATH = Path("C:\\Rohit\\Trainings\\repo\\FinancialAnalystCopilot\\database\\chroma").resolve()

print(f"Opening Chroma database at: {DB_PATH}")

client = chromadb.PersistentClient(
    path=str(DB_PATH),
    settings=Settings(anonymized_telemetry=False),
)

collections = client.list_collections()
collection_names = [collection.name for collection in collections]

print(f"Available collections: {collection_names}")

for collection in collections:
    collection = client.get_collection(name=collection.name)

    print(f"Collection: {collection.name}")
    print(f"Number of records: {collection.count()}")

    results = collection.get(
        limit=5,
        include=["documents", "metadatas", "embeddings"],
    )

    embeddings = results.get("embeddings")
    documents = results.get("documents")
    metadatas = results.get("metadatas")

    for index, record_id in enumerate(results["ids"]):
        vector = embeddings[index] if embeddings is not None else None
        document = documents[index] if documents is not None else None
        metadata = metadatas[index] if metadatas is not None else None

        print("\n" + "=" * 80)
        print(f"ID: {record_id}")
        print(f"Metadata: {metadata}")

        if document:
            print(f"Document preview: {document[:300]}")

        if vector is not None:
            vector_dimension = len(vector)
            vector_norm = sqrt(
                sum(float(value) * float(value) for value in vector)
            )

            first_values = [
                round(float(value), 6)
                for value in vector[:10]
            ]

            print(f"Embedding dimension: {vector_dimension}")
            print(f"Embedding norm: {vector_norm:.6f}")
            print(f"First 10 embedding values: {first_values}")

# for collection in collections:
#     collection = client.get_collection(name=collection.name)
#     #delete collection if it exists
#     #if collection.count() > 0:
#     print(f"Deleting collection: {collection.name}")
#     client.delete_collection(name=collection.name)

Opening Chroma database at: C:\Rohit\Trainings\repo\FinancialAnalystCopilot\database\chroma
Available collections: ['finance_docs_chunks', 'sample_docs_chunks']
Collection: finance_docs_chunks
Number of records: 176

ID: analyst_conversation_transcripts_q2_q3_2026.docx:0
Metadata: {'headline': 'Meeting 1: Q2 Close Review - Overview', 'word_count': 69, 'document_name': 'analyst_conversation_transcripts_q2_q3_2026.docx', 'summary': 'The meeting focused on reviewing synthetic finance results and preparing management commentary.', 'chunk_index': 0}
Document preview: Meeting 1: Q2 Close Review

Participants: Nathan Okafor, Jordan Ellis, Victor Alvarez, Mira Cole, Owen Hughes, Bethany Luo

Context: Meeting 1: Q2 Close Review was convened to review the synthetic finance results, prepare management commentary, and align on the questions analysts are likely to ask. 
Embedding dimension: 3072
Embedding norm: 1.000129+0.000000j
First 10 embedding values: [-0.00753, 0.019196, -0.021576, -0.001718,

In [ ]:
from services.rag.embeddings import OpenAIEmbeddingProvider

#answer question using the embeddings in the Chroma database
collection = client.get_or_create_collection("sample_docs_chunks")

question = "company name?"
#query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding

embedding_provider = OpenAIEmbeddingProvider()
query_embedding = embedding_provider.embed_texts([question])
#print(query_embedding)


query_results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

#MAX_DISTANCE = 0.5

for result in zip(query_results["documents"][0], query_results["metadatas"][0], distances := query_results["distances"][0]):
    #if result[2] <= MAX_DISTANCE:
    print(f"Distance: {result[2]}", f"Document: {result[0][:100]}...", f"Metadata: {result[1]}")

#print(query_results)


In [ ]:
#test retrieval from chroma
#write code to call the orchestrate_new function from the Orchestrator class in the services.orchestration.openAIOrchestration.orchestrator module, and pass a sample payload to it. Print the final output from the orchestrate_new function.
from services.orchestration.openAIOrchestration.orchestrator import Orchestrator, OrchestratorRequest

payload = OrchestratorRequest(user_question = "Give me a high level on most recent quaterly results")

async def test_orchestrate_new(payload):
    print(payload)
    _openAI_Orchestrator: Orchestrator = Orchestrator()
    async for output in _openAI_Orchestrator.orchestrate_new(payload):
        print(output)

await test_orchestrate_new(payload)

In [ ]:
import json
import requests

question = "Summarize the relevant document context and give me the budget trend for Q3."
api_base_url = "http://127.0.0.1:8000"

print("Calling OpenAI orchestrator endpoint...")
response = requests.post(
    f"{api_base_url}/orchestrator/ask-openai",
    json={
        "user_question": que
        stion,
        "api_base_url": api_base_url,
    },
    timeout=120,
    stream=True,
)

print(f"HTTP status: {response.status_code}")
print("Content-Type:", response.headers.get("content-type"))

for line in response.iter_lines(decode_unicode=True):
    if not line:
        continue
    print(line)


In [12]:
# Clear / purge all existing Chroma collections
from pathlib import Path
#import chromadb
from chromadb.config import Settings

#DB_PATH = Path("C:\\Rohit\\Trainings\\repo\\FinancialAnalystCopilot\\database\\chroma").resolve()
#clientForDeletion = chromadb.PersistentClient(path=str(DB_PATH), settings=Settings(anonymized_telemetry=False))

collections = client.list_collections()
collection_names = [collection.name for collection in collections]

print(f"Found {len(collection_names)} collection(s) in: {DB_PATH}")

# if collection_names:
#     for name in collection_names:
#         print(f"Deleting collection: {name}")
#         client.delete_collection(name)

#     print("All collections have been purged.")
# else:
#     print("No collections found to purge.")


NameError: name 'DB_PATH' is not defined

In [ ]:
# Test the OpenAI orchestrator with a sample user question.
# This guards against stale imports when the notebook kernel has already loaded
# a conflicting `services` module.
import sys
from pathlib import Path

repo_root = find_repo_root(Path.cwd()) if 'find_repo_root' in globals() else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

for name in [n for n in list(sys.modules) if n == 'services' or n.startswith('services.')]:
    sys.modules.pop(name, None)

from services.orchestration.openAIOrchestration.orchestrator import Orchestrator
from services.models.models import OrchestratorRequest

user_request = OrchestratorRequest(
    user_question="search within the document to find the name of the company"
)

orchestrator_instance = Orchestrator()
result = await orchestrator_instance.orchestrate_new(user_request)
#print(result)

In [11]:
#write code to call query_chroma_collection_new in chroma.py, and pass a sample query and metadata filters to it. Print the results returned by the function.
from services.routers.chroma import query_chroma_collection_new
from services.routers.chroma import ChromaQueryRequestParameters

sample_query = "name of analysts in the conversations"
sample_metadata_filters = {}

arguments = ChromaQueryRequestParameters(
    query=sample_query,
    metadata_filters=sample_metadata_filters
)

results = query_chroma_collection_new(arguments)

#print a well formatted output of the results returned by the function
import json

print(json.dumps(results, indent=4))


{
    "ids": [
        [
            "analyst_conversation_transcripts_q2_q3_2026.docx:0",
            "q3_2026_guidance_growth_prospects_expanded.docx:21",
            "analyst_conversation_transcripts_q2_q3_2026.docx:3",
            "analyst_conversation_transcripts_q2_q3_2026.docx:20",
            "company_profile_stakeholder_overview.docx:5",
            "q3_2026_guidance_growth_prospects_expanded.docx:41",
            "q3_2026_guidance_growth_prospects_expanded.docx:39",
            "q3_2026_guidance_growth_prospects_expanded.docx:37",
            "stakeholder_interview_notes_finance_leadership.docx:0",
            "tailwinds_headwinds_driver_detail_dossier.docx:1",
            "company_profile_stakeholder_overview.docx:3",
            "company_stakeholder_operating_context_deepdive.docx:10",
            "upcoming_growth_drivers_management_commentary.docx:12",
            "analyst_conversation_transcripts_q2_q3_2026.docx:11",
            "upcoming_growth_drivers_management_comment

In [9]:
from chromadb import PersistentClient
from chromadb.config import Settings
from services.routers.chroma import ChromaQueryResult

#validate chunk result
DEFAULT_CHROMA_DB_PATH = "C:\\Rohit\\Trainings\\repo\\FinancialAnalystCopilot\\database\\chroma"
#Path(os.getenv("CHROMA_PERSIST_DIR", "")) if os.getenv("CHROMA_PERSIST_DIR") else Path(__file__).resolve().parents[2] / "database" / "chroma"

client = PersistentClient(path=str(DEFAULT_CHROMA_DB_PATH), settings=Settings(anonymized_telemetry=False))

collection_name: str = "finance_docs_chunks"
collection_names = [collection.name for collection in client.list_collections()]
if collection_name not in collection_names:
    raise ValueError(f"Collection '{collection_name}' not found. Available collections: {collection_names}")

collection = client.get_collection(name=collection_name)


chunk = collection.get(
    ids="q2_2026_finance_performance_report_expanded.docx:16",
    include=["documents", "embeddings"],
)

stored_text = chunk["documents"][0]
stored_embedding = chunk["embeddings"][0]
saved_embedding = chunk["embeddings"]

print(f"Stored text for chunk 'q2_2026_finance_performance_report_expanded.docx:16': {stored_text[:200]}...")
print(f"Stored embedding for chunk 'q2_2026_finance_performance_report_expanded.docx:16': {stored_embedding[:10]}... (length: {len(stored_embedding)})")

#embed stored_text
query_embedding = ChromaQueryResult.embedding_provider.embed_texts([stored_text])
print(f"Query embedding for stored text: {query_embedding[0][:10]}... (length: {len(query_embedding[0])})")

import numpy as np

saved = np.array(saved_embedding, dtype=np.float64).flatten()
new = np.array(query_embedding, dtype=np.float64).flatten()

print("saved shape:", saved.shape)
print("new shape:", new.shape)

cosine_similarity = np.dot(saved, new) / (
    np.linalg.norm(saved) * np.linalg.norm(new)
)

cosine_distance = 1 - cosine_similarity

print("saved norm:", np.linalg.norm(saved))
print("new norm:", np.linalg.norm(new))
print("cosine similarity:", cosine_similarity)
print("cosine distance:", cosine_distance)
print("max abs difference:", np.max(np.abs(saved - new)))
print("mean abs difference:", np.mean(np.abs(saved - new)))

Stored text for chunk 'q2_2026_finance_performance_report_expanded.docx:16': Owen Hughes

How should the copilot frame favorable expense variance if spending is simply delayed?

Label it as a timing item unless there is clear evidence of permanent savings. For example, delayed...
Stored embedding for chunk 'q2_2026_finance_performance_report_expanded.docx:16': [-0.01412201 -0.01661682 -0.02049256  0.02714539 -0.00419998  0.01124573
 -0.03417969  0.02519226 -0.05569458 -0.01299286]... (length: 3072)
Query embedding for stored text: [-0.0141448974609375, -0.0166778564453125, -0.0205230712890625, 0.0271148681640625, -0.00412750244140625, 0.01128387451171875, -0.03411865234375, 0.0252227783203125, -0.05572509765625, -0.012969970703125]... (length: 3072)
saved shape: (3072,)
new shape: (3072,)
saved norm: 0.9997287099490509
new norm: 1.000156267706314
cosine similarity: 0.9999967554182022
cosine distance: 3.2445817977899694e-06
max abs difference: 0.00019073486328125
mean abs difference: 3.

In [10]:
saved = np.array(saved_embedding, dtype=np.float64).flatten()

results = collection.query(
    query_embeddings=[saved.tolist()],
    n_results=5,
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)

for chunk_id, distance, doc in zip(
    results["ids"][0],
    results["distances"][0],
    results["documents"][0],
):
    print("id:", chunk_id)
    print("distance:", distance)
    print(doc[:200])
    print()

id: q2_2026_finance_performance_report_expanded.docx:16
distance: 2.205371856689453e-06
Owen Hughes

How should the copilot frame favorable expense variance if spending is simply delayed?

Label it as a timing item unless there is clear evidence of permanent savings. For example, delayed

id: q3_2026_guidance_growth_prospects_expanded.docx:30
distance: 2.205371856689453e-06
Owen Hughes

How should the copilot frame favorable expense variance if spending is simply delayed?

Label it as a timing item unless there is clear evidence of permanent savings. For example, delayed

id: analyst_conversation_transcripts_q2_q3_2026.docx:30
distance: 0.07761728763580322
Exchange 2.10: Owen Hughes

Owen Hughes: How should the copilot frame favorable expense variance if spending is simply delayed?

Management response: Label it as a timing item unless there is clear ev

id: analyst_conversation_transcripts_q2_q3_2026.docx:13
distance: 0.08865690231323242
Exchange 1.10: Owen Hughes

Owen Hughes: How sh